# Урок 16. SQL: группировка и соединения

11 класс · II четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 15](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-15.ipynb) · [Урок 17 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-17.ipynb)

---

Агрегатные функции COUNT, SUM, AVG, MIN, MAX. GROUP BY и HAVING. Соединение таблиц JOIN. Запросы по нескольким таблицам.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-16", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### От «покажи строки» к «посчитай по группам»

Прошлый урок отвечал на вопросы вида «какие записи подходят».
Сегодня — вопросы другого сорта: сколько книг у каждого автора,
какой средний объём по жанрам, кто взял больше всех книг. Такие
запросы называют аналитическими, и в них появляются две новые вещи:
**агрегатные функции** и **группировка**.

### Агрегатные функции

| Функция | Что считает |
|---|---|
| `COUNT(*)` | количество строк |
| `COUNT(поле)` | количество непустых значений поля |
| `SUM(поле)` | сумму |
| `AVG(поле)` | среднее |
| `MIN(поле)`, `MAX(поле)` | наименьшее и наибольшее |

Агрегатная функция сворачивает много строк в одно значение:

```sql
SELECT COUNT(*), AVG(страниц) FROM книги
```

### GROUP BY

Если добавить `GROUP BY`, база сначала разложит строки по группам,
а потом применит функцию к каждой группе отдельно:

```sql
SELECT автор, COUNT(*)
FROM книги
GROUP BY автор
```

Получится по строке на автора. Правило, которое стоит запомнить
намертво: **в SELECT можно перечислять только поля из GROUP BY
и агрегатные функции**. Иначе непонятно, какое из значений группы
показывать.

### WHERE или HAVING

| Часть | Когда работает | Что фильтрует |
|---|---|---|
| `WHERE` | до группировки | отдельные строки |
| `HAVING` | после группировки | целые группы |

```sql
SELECT автор, COUNT(*) AS сколько
FROM книги
WHERE год > 1850          -- отбросили старые книги
GROUP BY автор
HAVING COUNT(*) >= 2      -- оставили авторов с двумя и более книгами
ORDER BY сколько DESC
```

Слово `AS` задаёт имя столбца результата — так на него можно
ссылаться в `ORDER BY` и удобнее читать вывод.

### Порядок частей запроса

```
SELECT … FROM … WHERE … GROUP BY … HAVING … ORDER BY … LIMIT …
```

Порядок жёсткий, менять части местами нельзя. А выполняется всё
в другом порядке: сначала FROM, потом WHERE, потом GROUP BY,
затем HAVING, и только под конец SELECT и ORDER BY. Поэтому
в `WHERE` нельзя пользоваться именем из `AS` — его тогда ещё
не существует.

### JOIN: собираем таблицы вместе

Данные разложены по таблицам, а вопрос обычно про всё сразу:
«кто какую книгу взял». Соединение делает `JOIN`:

```sql
SELECT читатели.фамилия, книги.название
FROM выдачи
JOIN читатели ON выдачи.читатель = читатели.id
JOIN книги    ON выдачи.книга    = книги.id
```

После `ON` пишется условие связи — обычно «внешний ключ равен
первичному ключу». Соединять можно сколько угодно таблиц подряд.

### INNER JOIN и LEFT JOIN

| Вид | Что возвращает |
|---|---|
| `JOIN` (он же `INNER JOIN`) | только строки, у которых есть пара |
| `LEFT JOIN` | все строки левой таблицы; где пары нет — `NULL` |

Разница принципиальная. «Сколько книг взял каждый читатель»
с обычным `JOIN` потеряет тех, кто не взял ни одной: у них просто
нет строк в таблице выдач. С `LEFT JOIN` они останутся, и `COUNT`
по полю выдачи даст для них ноль.

### Три частые ошибки

1. Забыли `ON` — база соединит каждую строку с каждой, и вместо
   десяти строк получится сто.
2. Написали `COUNT(*)` вместо `COUNT(поле)` при `LEFT JOIN`:
   `COUNT(*)` считает строки, а строка есть даже с `NULL`, и вместо
   нуля получится единица.
3. Условие на группу написали в `WHERE` — база откажется выполнять
   запрос: агрегатной функции там не место.

## Смотрим, как это работает

### Готовим базу библиотеки

In [ ]:
import sqlite3

соединение = sqlite3.connect(":memory:")
курсор = соединение.cursor()

курсор.execute("""CREATE TABLE книги (
    id INTEGER PRIMARY KEY, название TEXT, автор TEXT,
    год INTEGER, страниц INTEGER, жанр TEXT)""")
курсор.execute("""CREATE TABLE читатели (
    id INTEGER PRIMARY KEY, фамилия TEXT, класс TEXT)""")
курсор.execute("""CREATE TABLE выдачи (
    id INTEGER PRIMARY KEY, читатель INTEGER, книга INTEGER, дата TEXT)""")

курсор.executemany("INSERT INTO книги VALUES (?, ?, ?, ?, ?, ?)", [
    (1, "Мастер и Маргарита", "Булгаков", 1967, 480, "роман"),
    (2, "Собачье сердце", "Булгаков", 1987, 128, "повесть"),
    (3, "Преступление и наказание", "Достоевский", 1866, 672, "роман"),
    (4, "Идиот", "Достоевский", 1869, 640, "роман"),
    (5, "Му-му", "Тургенев", 1854, 32, "рассказ"),
    (6, "Отцы и дети", "Тургенев", 1862, 288, "роман"),
    (7, "Вишнёвый сад", "Чехов", 1904, 96, "пьеса"),
    (8, "Каштанка", "Чехов", 1887, 48, "рассказ"),
])
курсор.executemany("INSERT INTO читатели VALUES (?, ?, ?)", [
    (1, "Иванов", "11А"),
    (2, "Петрова", "10Б"),
    (3, "Сидоров", "11А"),
    (4, "Кузнецова", "10Б"),
])
курсор.executemany("INSERT INTO выдачи VALUES (?, ?, ?, ?)", [
    (1, 1, 1, "2026-09-12"),
    (2, 1, 2, "2026-09-20"),
    (3, 1, 5, "2026-10-01"),
    (4, 2, 1, "2026-09-25"),
    (5, 2, 3, "2026-10-05"),
    (6, 3, 7, "2026-10-08"),
])
соединение.commit()


def показать(запрос):
    строки = курсор.execute(запрос).fetchall()
    заголовки = [описание[0] for описание in курсор.description]
    print(" | ".join(заголовки))
    print("-" * 55)
    for строка in строки:
        print(" | ".join(str(значение) for значение in строка))
    print(f"[строк: {len(строки)}]\n")


показать("SELECT COUNT(*) AS всего_книг, AVG(страниц) AS средний_объём FROM книги")

### Пример 1. Группировка по автору

In [ ]:
показать("""
SELECT автор, COUNT(*) AS книг, SUM(страниц) AS страниц_всего
FROM книги
GROUP BY автор
ORDER BY книг DESC, автор
""")

Одна строка на автора. Сортировка по двум полям сразу решает,
что делать при равенстве: сначала по количеству книг, при равном —
по алфавиту.

### Пример 2. Средний объём по жанрам

In [ ]:
показать("""
SELECT жанр, COUNT(*) AS книг, ROUND(AVG(страниц), 1) AS средний
FROM книги
GROUP BY жанр
ORDER BY средний DESC
""")

`ROUND(значение, знаков)` округляет — без него среднее выводится
со всеми знаками после запятой.

### Пример 3. WHERE против HAVING

In [ ]:
print("Только книги после 1860 года, авторы с двумя и более книгами:")
показать("""
SELECT автор, COUNT(*) AS книг
FROM книги
WHERE год > 1860
GROUP BY автор
HAVING COUNT(*) >= 2
ORDER BY автор
""")

`WHERE` выбросил «Му-му» 1854 года ещё до группировки, поэтому
у Тургенева осталась одна книга — и `HAVING` убрал его целиком.

### Пример 4. Соединение таблиц

In [ ]:
показать("""
SELECT читатели.фамилия, книги.название, выдачи.дата
FROM выдачи
JOIN читатели ON выдачи.читатель = читатели.id
JOIN книги    ON выдачи.книга    = книги.id
ORDER BY выдачи.дата
""")

### Пример 5. JOIN вместе с группировкой

Сколько книг взял каждый читатель?

In [ ]:
print("Обычный JOIN:")
показать("""
SELECT читатели.фамилия, COUNT(*) AS взял
FROM выдачи
JOIN читатели ON выдачи.читатель = читатели.id
GROUP BY читатели.фамилия
ORDER BY взял DESC
""")

print("LEFT JOIN — видно и тех, кто ничего не брал:")
показать("""
SELECT читатели.фамилия, COUNT(выдачи.id) AS взял
FROM читатели
LEFT JOIN выдачи ON выдачи.читатель = читатели.id
GROUP BY читатели.фамилия
ORDER BY взял DESC, читатели.фамилия
""")

В первом запросе Кузнецовой нет вовсе: она не брала книг, и строк
в таблице выдач для неё не существует. Во втором она есть с нулём —
именно потому, что мы посчитали `COUNT(выдачи.id)`, а не `COUNT(*)`.

### Пример 6. Самая популярная книга

In [ ]:
показать("""
SELECT книги.название, COUNT(*) AS раз_выдана
FROM выдачи
JOIN книги ON выдачи.книга = книги.id
GROUP BY книги.название
ORDER BY раз_выдана DESC, книги.название
LIMIT 3
""")

Такой запрос — готовый ответ на вопрос школьного библиотекаря
«что закупать в следующий раз». Ровно эти же три строки SQL стоят
за разделом «популярное» в любом интернет-магазине.

## Пробуем сами

Работаем с таблицами `книги`, `читатели`, `выдачи`.

### Задача 1. Книг у каждого автора

Верните пары «автор, количество книг», по убыванию количества,
при равенстве — по алфавиту.

In [ ]:
def книг_у_автора():
    return ...

In [ ]:
si.check("1", книг_у_автора, [
    ((), [("Булгаков", 2), ("Достоевский", 2), ("Тургенев", 2), ("Чехов", 2)]),
])

### Задача 2. Самая толстая книга каждого жанра

Верните «жанр, наибольшее число страниц», по алфавиту жанров.

In [ ]:
def максимум_по_жанрам():
    return ...

In [ ]:
si.check("2", максимум_по_жанрам, [
    ((), [("повесть", 128), ("пьеса", 96), ("рассказ", 48), ("роман", 672)]),
])

### Задача 3. Только большие жанры

Верните жанры, в которых больше одной книги, и количество книг.
По алфавиту жанра.

In [ ]:
def жанры_с_несколькими():
    return ...

In [ ]:
si.check("3", жанры_с_несколькими, [
    ((), [("рассказ", 2), ("роман", 4)]),
])

### Задача 4. Кто что взял

Верните пары «фамилия, название» для всех выдач, упорядоченные
по фамилии, а при равенстве — по названию.

In [ ]:
def кто_что_взял():
    return ...

In [ ]:
si.check("4", кто_что_взял, [
    ((), [("Иванов", "Мастер и Маргарита"),
          ("Иванов", "Му-му"),
          ("Иванов", "Собачье сердце"),
          ("Петрова", "Мастер и Маргарита"),
          ("Петрова", "Преступление и наказание"),
          ("Сидоров", "Вишнёвый сад")]),
])

### Задача 5. Никто не забыт

Верните «фамилия, сколько книг взял» для **всех** читателей, включая
тех, кто не брал ничего. По убыванию количества, при равенстве —
по фамилии.

In [ ]:
def все_читатели():
    return ...

In [ ]:
si.check("5", все_читатели, [
    ((), [("Иванов", 3), ("Петрова", 2), ("Сидоров", 1), ("Кузнецова", 0)]),
])

### Задача 6. Активность классов

Верните «класс, количество выдач» — сколько книг взяли ученики
каждого класса. Только классы, где выдачи были. По убыванию
количества, при равенстве — по названию класса.

In [ ]:
def по_классам():
    return ...

In [ ]:
si.check("6", по_классам, [
    ((), [("11А", 4), ("10Б", 2)]),
])

### Задача 7. Где фильтровать группы

Каким словом отбирают **группы** после группировки?

In [ ]:
#@title 🧩 Задача 7. Фильтр групп { display-mode: "form" }
#@markdown Выберите ответ
слово = "выбери ответ" #@param ["выбери ответ", "WHERE", "HAVING", "GROUP BY"]

si.ответ("7", слово, "2f2e601f101ec3d8",
         hint="WHERE работает до группировки.")

## Домашнее задание

### Домашнее задание 1. Средний год по жанрам

Верните «жанр, средний год издания, округлённый до целого»
по алфавиту жанра.

In [ ]:
def средний_год():
    return ...

In [ ]:
si.check("дз1", средний_год, [
    ((), [("повесть", 1987), ("пьеса", 1904), ("рассказ", 1871), ("роман", 1891)]),
])

### Домашнее задание 2. Книги, которые никто не брал

Верните названия книг, ни разу не выданных, по алфавиту.
Понадобится `LEFT JOIN` и условие на `NULL`.

In [ ]:
def невостребованные():
    return ...

In [ ]:
si.check("дз2", невостребованные, [
    ((), [("Идиот",), ("Каштанка",), ("Отцы и дети",)]),
])

### Домашнее задание 3. Аналитика своей базы

Возьмите свою базу с прошлого урока и напишите восемь аналитических
запросов: минимум три с `GROUP BY`, два с `HAVING`, два с `JOIN`
и один с `LEFT JOIN`, который показывает «пустые» записи. К каждому
подпишите вопрос по-русски и объясните, почему выбрали именно такой
вид соединения.

---

### Любопытно

Соединение двух таблиц по миллиону строк каждая — операция, которую
наивно выполнить нельзя: перебор пар дал бы триллион сравнений.
Базы применяют хитрости: строят по одной таблице хеш-таблицу,
сортируют обе и идут навстречу, пользуются индексами. Выбор способа
делает оптимизатор, и на больших данных разница между удачным
и неудачным планом — минуты против суток.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 15](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-15.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 17 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-17.ipynb)